# React — Refs

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> This topic is short and mostly playground work. A ref's whole purpose is to reach a real DOM
> node, and there is no DOM in a notebook cell.

## LESSON 49 — Refs to DOM nodes

Everything so far has been React putting things on screen for you. This lesson is the first
time you reach past it and touch the page directly — and React is explicit about why that
exists:

> React automatically updates the DOM to match your render output, so your components won't
> often need to manipulate it. However, sometimes you might need access to the DOM elements
> managed by React — for example, to focus a node, scroll to it, or measure its size and
> position. There is no built-in way to do those things in React, so you will need a *ref* to
> the DOM node.

Focus, scroll, measure. Notice what those three have in common: **none of them is a value you
could put on screen.** You cannot render "this input is focused" or "this box is 301 pixels
wide" — focus is something you *do*, and size is something only the browser knows after it has
laid the page out.

That is the test for whether you need a ref. Not "I want to change the page" — React does that
— but "I need to ask the browser something, or tell it to do something, that has no
representation in my JSX".

### The three steps

```jsx
import { useRef } from "react";

function Form() {
  const inputRef = useRef(null);          // 1. declare

  function handleClick() {
    inputRef.current.focus();             // 3. use the node
  }

  return (
    <>
      <input ref={inputRef} />            {/* 2. attach */}
      <button onClick={handleClick}>Focus the input</button>
    </>
  );
}
```

> The `useRef` Hook returns an object with a single property called `current`. Initially,
> `myRef.current` will be `null`. When React creates a DOM node for this `<div>`, React will
> put a reference to this node into `myRef.current`.

`useRef` is a Hook, so LESSON 37 applies: top level of the component, never in a condition or
a loop.

### When `current` is actually filled in

This is the detail that causes the "it's null" confusion, and it is LESSON 38's vocabulary
again:

> React sets `ref.current` during the commit. Before updating the DOM, React sets the affected
> `ref.current` values to `null`. After updating the DOM, React immediately sets them to the
> corresponding DOM nodes.

So during the **first render** there is no node yet and `current` is `null`. By the time an
event handler or an Effect runs, the commit has happened and the node is there. Reading
`inputRef.current` in the component body on the first render gets you `null`, every time — and
the playground experiment logs exactly that so you can watch it.

Handlers and Effects are safe. The render itself is not.

### Reaching a child's node — `ref` is just a prop now

A component does not expose its DOM node by default. In React 19 you opt in by accepting `ref`
like any other prop:

```jsx
function SearchBox({ ref, label }) {
  return <label>{label}: <input ref={ref} /></label>;
}

// the parent
<SearchBox ref={childRef} label="Search" />
```

Measured in the playground: clicking the parent's button focuses the child's input, with no
warning in the console.

> **If you read older code** you will find `forwardRef` wrapped around components to do this.
> It still exists and still works, but it is no longer how new code is written — React's own
> documentation now shows `function MyInput({ ref })` directly. You do not need to learn
> `forwardRef` to write React today; you need to recognise it.

There is a nice symmetry with LESSON 20 here. `key` is React's own bookkeeping and never
reaches your props. `ref` in React 19 **is** an ordinary prop and does reach them.

### The rule that keeps this safe

> Refs are an escape hatch. Manually manipulating *another* component's DOM nodes can make your
> code fragile.

And more bluntly:

> Avoid changing DOM nodes managed by React.

Reading is safe — focus, scroll, measure, select. **Changing** is not. If you remove a node by
hand, or add children to one React is rendering, you have made React's picture of the page
wrong, and the docs say what follows:

> After you've manually removed the DOM element, trying to use `setState` to show it again will
> lead to a crash. This is because you've changed the DOM, and React doesn't know how to
> continue managing it correctly.

The safe boundary: use a ref to **ask** the DOM something or to trigger a browser behaviour.
Use state to **change what is on screen**.

### Key Notes

- A ref gives you the real DOM node, for the things JSX cannot express: focus, scroll, measure.
- `useRef(null)` → attach with `ref={…}` → read `current` in a handler or Effect.
- `current` is filled in **during the commit**, so it is `null` during the first render.
- In React 19 `ref` is an ordinary prop; `forwardRef` is for reading old code, not writing new.
- Read the DOM, don't rewrite it. Changing nodes React manages breaks React.

### Example

**In the playground.** There is no cell — a ref points at a DOM node, and there is no DOM here.
Writing a fake one would teach you the fake.

Point `playground/src/App.jsx` at `./experiments/22-refs.jsx` and open the console. Reload and
read the very first `render` line before you click anything: it reports whether
`inputRef.current` is `null` at that moment. Then focus the input, measure the box, and focus
the child's input from the parent's button.

### Exercise

**In the playground**, in `22-refs.jsx`.

1. Reload with the console open and read the first `render` line. What is `inputRef.current`
   during the first render, and which step of LESSON 38 has not happened yet?
2. Click **measure the box**, then resize the browser window and click it again. Does the
   number change? Now explain why this value could not have been kept in state instead.
3. Add a third button, **scroll to the box**, using `boxRef.current.scrollIntoView()`. Make the
   page tall enough to scroll first — a `<div style={{ height: "150vh" }} />` above the box
   will do.
4. Try to read `inputRef.current` **in the component body** (not in a handler) and log its
   `value` — for example `console.log(inputRef.current.value)`. Reload. What happens, and what
   is the smallest change that makes it safe?
5. In `SearchBox`, rename the `ref` prop to `inputRef` in both the child's parameters and the
   parent's JSX. What happens, and what does that tell you about whether `ref` is special?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

For each, say whether a **ref** is the right tool, or whether **state** is — and why.

1. Moving focus to the first field with an error after a failed submit.
2. Showing a "copied!" message for two seconds after the user copies a code.
3. Scrolling a chat window to the bottom when a new message arrives.
4. Remembering whether a panel is open.
5. Knowing how tall a text area has grown so a sibling can match it.
6. Playing a `<video>` when the user clicks a custom button.
7. Highlighting the currently selected row in a table.

Then answer:

- Two of these are the same shape: something the browser owns that you are *asking* or
  *telling*, rather than something you are rendering. Which two, and what is the phrase from
  the lesson that identifies them?
- One of them tempts people into a ref because "it's not really UI state". Which, and what goes
  wrong if you use a ref for it?

In [ ]:
// Your code here

## LESSON 50 — Refs that hold values

A ref does not have to point at a DOM node. React's definition of the Hook does not mention
the DOM at all:

> `useRef` is a React Hook that lets you reference a value that's **not needed for rendering**.

That phrase is the entire lesson.

### The missing half of LESSON 25

LESSON 25 explained why a plain variable cannot hold state, and gave **two** reasons:

> 1. **Retain** the data between renders.
> 2. **Trigger** React to render the component with new data.

A plain variable does neither. State does both. A ref does **exactly one**:

| | survives a render? | triggers a render? |
|---|---|---|
| `let count = 0` in the body | no | no |
| `useRef(0)` | **yes** | no |
| `useState(0)` | **yes** | **yes** |

> You can **store information** between re-renders (unlike regular variables, which reset on
> every render).

> **Changing a ref does not trigger a re-render.** This means refs are perfect for storing
> information that doesn't affect the visual output of your component.

So a ref is for a value you need to *keep* but never *show*. If the value appears on screen, it
is state. If it does not, a ref is lighter and cannot cause a render loop.

### Use 1: a timer id

The classic case, and it is the one you could not have written before:

```jsx
const intervalRef = useRef(null);

useEffect(() => {
  if (!running) return;

  intervalRef.current = setInterval(() => setElapsed((e) => e + 1), 100);

  return () => clearInterval(intervalRef.current);
}, [running]);
```

Think about the alternatives and why both fail:

- **a plain variable** — recreated on every render, so by the time the cleanup runs the id it
  was holding is gone and `clearInterval(undefined)` does nothing. The interval runs forever.
- **state** — setting it would trigger a render, which is pointless work for a number nobody
  sees, and it makes the Effect's dependencies awkward.

The id needs to survive and must not cause renders. That is a ref, precisely.

### Use 2: the previous value

React gives you the current value. Sometimes you want the one before it — to animate a
difference, to detect a direction, to log a change:

```jsx
const previousCount = useRef(null);

useEffect(() => {
  previousCount.current = count;
}, [count]);
```

The write happens in an Effect, so it runs *after* the commit. During a render `count` is the
new value and `previousCount.current` is still the old one — which is exactly the lag you
wanted. Measured in the playground:

```text
click +1   ->  count: 1 · previously: 0
click +1   ->  count: 2 · previously: 1
```

One render behind, reliably.

### The proof that it does not render

The playground has a counter kept in a ref instead of state. Click it twice and the console
says the ref really did change:

```text
   brokenRef is now 1 — but nothing re-rendered
   brokenRef is now 2 — but nothing re-rendered
```

and the screen still reads **0**. The value is genuinely 2. React was never told, so the
component was never called again, so nothing on screen could possibly change.

That is not a bug in the experiment — it is the defining property of a ref, shown as the
failure it becomes when you use one for something that renders.

### Key Notes

- A ref holds a value that **survives renders but never causes one** — the half of `useState`
  that remembers, without the half that re-renders.
- Use it for things the user never sees: timer ids, subscription handles, a previous value, a
  "has this already run" flag.
- If the value appears on screen, it must be state. A ref will change and the screen will not.
- Write a previous-value ref in an **Effect**, so it updates after the commit and lags one
  render behind.

### Example

**In the playground.** No cell — every use here is about surviving across renders, and a
notebook cell has no renders to survive.

Point `playground/src/App.jsx` at `./experiments/23-ref-values.jsx`. Run the stopwatch, click
`+1` a few times and watch "previously" trail behind, then click the ref counter twice and
watch the screen refuse to move.

### Exercise

**In the playground**, in `23-ref-values.jsx`.

1. Start the stopwatch, then stop it. Read the two console lines. Now **replace
   `intervalRef` with a plain `let intervalId = null;`** declared in the component body, start
   and stop it, and describe what goes wrong — and why the console's "stopped" line is
   misleading.
2. Put the ref back. Now change `intervalRef` to `useState(null)` instead, storing the id in
   state. It will appear to work. Look at the console and count the renders — what is the cost,
   and what happens if you add `intervalId` to the Effect's dependency array?
3. Fix the broken ref counter so it works, changing as little as possible. What did you change,
   and which of the two properties from the table did you need?
4. Add a second ref that counts **how many times the component has rendered**, incremented in
   an Effect with no dependency array. Display it. Then explain why you cannot increment it
   during render instead — LESSON 51 is about that, so answer from what you already know.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

For each value, say **ref** or **state**, in a few words.

1. The id returned by `setTimeout` for a toast that auto-dismisses.
2. The number of seconds left on that toast, shown as a countdown.
3. Whether the user has already been shown a one-time tooltip this session.
4. The text currently typed into a search box.
5. A `WebSocket` instance the component opened.
6. Whether the websocket is currently connected, shown as a coloured dot.
7. The scroll position to restore when the user comes back to a list.

Then answer: numbers 5 and 6 are about the same connection and have different answers. Explain
why in one sentence, using the phrase from the lesson.

In [ ]:
// Your code here

## LESSON 51 — Ref or state, and the one rule

Two lessons of refs come down to one decision and one rule.

### The decision

> Information that's used for rendering should be state instead.

That is the whole test, and it is shorter than any flowchart:

```text
Does this value decide what appears on screen?
    yes  ->  state
    no   ->  ref  (and only if it must survive renders — otherwise a plain variable)
```

Apply it honestly. "Not really UI state" is the phrase people use just before getting this
wrong; if the value is read anywhere in the JSX, or decides a branch that is, it is state,
however internal it feels.

| | |
|---|---|
| **state** | the user sees it, or it decides what they see |
| **ref** | you need it later, and the user never learns it exists |
| **plain variable** | you need it for the length of one render and no longer |

That third row matters. Not everything needs a Hook — a value computed and used within a
single render is just a `const` (LESSON 29).

### The rule: not during render

> **Do not write *or read* `ref.current` during rendering.**
>
> ```js
> function MyComponent() {
>   // 🚩 Don't write a ref during rendering
>   myRef.current = 123;
>   // 🚩 Don't read a ref during rendering
>   return <h1>{myOtherRef.current}</h1>;
> }
> ```
>
> You can read or write refs **from event handlers or effects instead**.

### Why the rule exists

Not arbitrary strictness — it follows from something you already know:

> React expects that the body of your component behaves like a pure function:
> - If the inputs (props, state, and context) are the same, it should return exactly the same
>   JSX.
> - Calling it in a different order or with different arguments should not affect the results
>   of other calls.
>
> Reading or writing a ref **during rendering** breaks these expectations.

A ref is mutable and invisible to React. Write to it during render and your component's output
now depends on how many times it was called — and React calls components more often than you
think, as LESSON 41's Strict Mode double-invoke demonstrated. Read from it during render and
you are rendering from a value React has no idea changed, so the screen can show something
stale with no way to correct itself.

React does not enforce this. There is no warning, no error, and the playground's broken counter
reads `brokenRef.current` in its JSX with a completely clean console. **It is a rule you keep,
not a rule that keeps you** — which is exactly why it is worth understanding rather than
memorising.

> Strict Mode makes impurity easier to catch, not impossible to write:
> *"In Strict Mode, React will call your component function twice in order to help you find
> accidental impurities."*

### Where it is safe

| | safe? | why |
|---|---|---|
| in an event handler | **yes** | runs after the commit, not during render |
| in an Effect | **yes** | same — LESSON 39's timing |
| in the cleanup | **yes** | also after a commit |
| in the component body | **no** | this is the render |
| in the JSX | **no** | also the render |

### Key Notes

- If a value decides what is on screen it is **state**; if it must survive and stay invisible
  it is a **ref**; if it only matters this render it is a plain `const`.
- **Never read or write `ref.current` during render** — only in handlers, Effects and cleanups.
- The reason is purity: the component body must give the same JSX for the same inputs.
- Nothing enforces the rule. Breaking it produces stale screens, not errors.

### Example

**In the playground.** No cell. This lesson is a decision and a rule, and both are about
behaviour across renders that a notebook has none of.

Re-open `./experiments/23-ref-values.jsx` and look at the broken counter with fresh eyes: it
breaks **both** halves of this lesson at once. It uses a ref for something that renders, and it
reads `brokenRef.current` in the JSX. One component, two mistakes, zero warnings.

### Exercise

Answer in comments — this lesson is judgement, so the exercise is judging.

**Part 1 — classify.** For each, say state, ref, or plain variable, and give a one-line reason.

1. A `formattedTotal` string built from two props, used once in the JSX.
2. Whether a modal is open.
3. The `AbortController` for the request currently in flight.
4. How many times the user has clicked a button, shown as "clicked 3 times".
5. How many times the user has clicked a button, sent to analytics on unmount only.
6. A debounce timer's id.
7. The filtered list shown to the user.

Numbers 4 and 5 are the same fact with different answers. Say why.

**Part 2 — find the breakages.** Each of these breaks the rule. Say what goes wrong and where
the line should move to.

```jsx
// A
function Chart({ data }) {
  const renderCount = useRef(0);
  renderCount.current = renderCount.current + 1;
  return <p>Rendered {renderCount.current} times</p>;
}

// B
function Timer() {
  const startedAt = useRef(null);
  if (startedAt.current === null) startedAt.current = Date.now();
  return <p>Started at {startedAt.current}</p>;
}

// C
function Row({ item }) {
  const lastItem = useRef(item);
  const changed = lastItem.current.id !== item.id;
  lastItem.current = item;
  return <li className={changed ? "flash" : ""}>{item.label}</li>;
}
```

**Part 3 — in the playground.** In `23-ref-values.jsx`, make the broken counter display
correctly *without* deleting the ref: keep `brokenRef` as it is, and add whatever is needed so
the screen keeps up. Then say what you have actually built and whether you would ship it.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

A colleague has a list that flickers. They have tried putting the scroll position in state and
the app got slower; they moved it to a ref and now the scroll restores to the wrong place
sometimes. They ask which one is correct.

Answer in comments:

1. Why did state make it slower? Be specific about what happens on every scroll event.
2. Why does a ref restore the wrong position "sometimes"? Name the moment the value is read
   and the moment it is written.
3. What is the actual answer — and what does it tell you about questions of the form "should
   this be state or a ref?"
4. They then suggest reading the ref during render "just to check it". Using this lesson,
   explain what could go wrong even though nothing will warn them.

In [ ]:
// Your code here